In [3]:
import json

with open('splitted_data.json', 'r') as f:
     data = json.load(f)

In [10]:
from collections import defaultdict

def extract_organized_articles_clustered(data, x_tolerance=0.15):
    """
    Extract articles sorted by columns. Final articles are sorted by their
    topmost-leftmost block, with x_tolerance grouping for column detection.
    """
    raw_articles = []
    
    for page in data['test']:
        img_path = page['img_path']
        labels = page['labels']
        page_articles = page.get('articles', {})
        
        index_to_block = {block['index']: block for block in labels}
        
        for article_id, block_indices in page_articles.items():
            article_blocks = []
            for idx in block_indices:
                if idx in index_to_block:
                    article_blocks.append(index_to_block[idx])
            
            # Group blocks into columns based on X position
            columns = defaultdict(list)
            for block in article_blocks:
                x1 = block['box_coord']['x1']
                col_key = round(x1 / x_tolerance)
                columns[col_key].append(block)
            
            # Sort each column by Y (top to bottom), then combine columns left-to-right
            sorted_blocks = []
            for col_key in sorted(columns.keys()):
                col_blocks = sorted(columns[col_key], key=lambda b: b['box_coord']['y1'])
                sorted_blocks.extend(col_blocks)
            
            titles = [b for b in sorted_blocks if b.get('box_type') == 'title']
            texts = [b for b in sorted_blocks if b.get('box_type') == 'plain text']
            
            # Determine article position for sorting:
            # Use highest (smallest y) and most-left (smallest x) title block,
            # or if no title, use highest most-left text block
            primary_block = None
            if titles:
                primary_block = min(titles, key=lambda b: (b['box_coord']['x1'], b['box_coord']['y1']))
            else:
                primary_block = min(texts, key=lambda b: (b['box_coord']['x1'], b['box_coord']['y1']))
            
            sort_x = primary_block['box_coord']['x1'] if primary_block else 0
            sort_y = primary_block['box_coord']['y1'] if primary_block else 0
            # Group articles into columns using x_tolerance
            col_group = round(sort_x / x_tolerance) if primary_block else 0

            raw_articles.append({
                'img_path': img_path,
                'article_id': article_id,
                'title': [b['text_corrected'] for b in titles],
                'text': '\n'.join([b['text_corrected'] for b in texts]),
                '_page': img_path,
                '_col': col_group,
                '_y': sort_y,
                '_x': sort_x
            })
    
    # Sort final articles: 
    # 1. By page
    # 2. By column group (using x_tolerance)
    # 3. By Y position within column (top to bottom)
    # 4. By X within same Y (tie-breaker)
    raw_articles.sort(key=lambda a: (a['_page'], a['_col'], a['_y'], a['_x']))
    
    final_articles = []
    for article in raw_articles:
        final_articles.append({
            'img_path': article['img_path'],
            'article_id': article['article_id'],
            'title': article['title'],
            'text': article['text']
        })
    
    return final_articles

In [11]:
result = extract_organized_articles_clustered(data)

In [ ]:
import os

pages_texts = []
text = []
page = ''

path = 'MD'
os.makedirs(path,exist_ok=True)

for article in result:
    if article['img_path'] != page:
        if text:

            pages_texts.append({page: text})
            with open(f'{path}/{page}.md','w') as f:
                f.write('\n\n'.join(text))

        page = article['img_path']
        text = []

    text.append(f"# {'\n## '.join(article['title'])} \n\n{article['text']}")

with open(f'{path}/{page}.md','w') as f:
    f.write('\n\n'.join(text))


## PereStruct

In [1]:
import json

with open('PereStruct//results/final_result.json', 'r') as f:
    test_results = json.load(f)

In [16]:
from collections import defaultdict

def extract_organized_articles_clustered(data, x_tolerance=0.15):
    """
    Extract articles from prediction JSON where each block has article_id.
    Final articles are sorted by their topmost-leftmost block.
    """
    raw_articles = []
    
    for page in data:
        img_path = page['img_path']
        labels = page['labels']
        
        # Group blocks by article_id (instead of using page['articles'])
        articles_blocks = defaultdict(list)
        for block in labels:
            article_id = block.get('article_id')
            if article_id is not None:  # Skip unassigned blocks
                articles_blocks[article_id].append(block)
        
        # Process each article
        for article_id, article_blocks in articles_blocks.items():
            # Group blocks into columns based on X position
            columns = defaultdict(list)
            for block in article_blocks:
                x1 = block['box_coord']['x1']
                col_key = round(x1 / x_tolerance)
                columns[col_key].append(block)
            
            # Sort each column by Y (top to bottom), then combine columns left-to-right
            sorted_blocks = []
            for col_key in sorted(columns.keys()):
                col_blocks = sorted(columns[col_key], key=lambda b: b['box_coord']['y1'])
                sorted_blocks.extend(col_blocks)
            
            titles = [b for b in sorted_blocks if b.get('box_type') == 'title']
            texts = [b for b in sorted_blocks if b.get('box_type') == 'plain text']
            
            # Determine article position for sorting:
            # Use highest (smallest y) and most-left (smallest x) title block,
            # or if no title, use highest most-left text block
            primary_block = None
            if titles:
                primary_block = min(titles, key=lambda b: (b['box_coord']['x1'], b['box_coord']['y1']))
            else:
                primary_block = min(texts, key=lambda b: (b['box_coord']['x1'], b['box_coord']['y1']))
            
            sort_x = primary_block['box_coord']['x1'] if primary_block else 0
            sort_y = primary_block['box_coord']['y1'] if primary_block else 0
            # Group articles into columns using x_tolerance
            col_group = round(sort_x / x_tolerance) if primary_block else 0

            raw_articles.append({
                'img_path': img_path,
                'article_id': article_id,
                'title': [b['text_corrected'] for b in titles],
                'text': '\n'.join([b['text_corrected'] for b in texts]),
                '_page': img_path,
                '_col': col_group,
                '_y': sort_y,
                '_x': sort_x
            })
    
    # Sort final articles: 
    # 1. By page
    # 2. By column group (using x_tolerance)
    # 3. By Y position within column (top to bottom)
    # 4. By X within same Y (tie-breaker)
    raw_articles.sort(key=lambda a: (a['_page'], a['_col'], a['_y'], a['_x']))
    
    final_articles = []
    for article in raw_articles:
        final_articles.append({
            'img_path': article['img_path'],
            'article_id': article['article_id'],
            'title': article['title'],
            'text': article['text']
        })
    
    return final_articles


In [17]:
test_result = extract_organized_articles_clustered(test_results)

In [20]:
import os

pages_texts = []
text = []
page = ''

path = 'PereStruct'
os.makedirs(path,exist_ok=True)

for article in test_result:
    if article['img_path'] != page:
        if text:

            pages_texts.append({page: text})
            with open(f'{path}/{page.split('/')[1]}.md', 'w') as f:
                f.write('\n\n'.join(text))

        page = article['img_path']
        text = []

    text.append(f"# {'\n## '.join(article['title'])} \n\n{article['text']}")

with open(f'{path}/{page.split('/')[1]}.md', 'w') as f:
    f.write('\n\n'.join(text))

In [16]:
import os
import time
import numpy as np
import re
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score.rouge_scorer import RougeScorer

def clean_markdown(text: str) -> str:
            text = re.sub(r'!\[.*?\]\(.*?\)', '', text)      
            text = re.sub(r'\[.*?\]\(.*?\)', '', text)        
            text = re.sub(r'[*_~`#]', '', text)              
            text = re.sub(r'^\s*[-*+]\s', '', text, flags=re.MULTILINE)  
            text = re.sub(r'\n\s*\n', '\n', text)
            text = text.replace('\n','')             
            return text.strip()

def calculate_md_metrics(ref_dir: str, pred_dir: str, print_results: bool = True):
    ref_files = set(os.listdir(ref_dir))
    pred_files = set(os.listdir(pred_dir))
    common_files = sorted(list(ref_files.intersection(pred_files)))
        
    if not common_files:
        raise ValueError("Dataset error")
        
    rouge_scorer = RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    smoothing = SmoothingFunction().method1
    
    results = []
    
    for fname in common_files:
        ref_path = os.path.join(ref_dir, fname)
        pred_path = os.path.join(pred_dir, fname)
        
        with open(ref_path, 'r', encoding='utf-8') as f:
            ref_text = clean_markdown(f.read())

        with open(pred_path, 'r', encoding='utf-8') as f:
            pred_text = clean_markdown(f.read())
            
        if not ref_text or not pred_text:
            bleu, r1, r2, rL = 0.0, 0.0, 0.0, 0.0
        else:
            ref_tokens = ref_text.split()
            pred_tokens = pred_text.split()
            
            bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)
            
            rouge_scores = rouge_scorer.score(ref_text, pred_text)
            r1 = rouge_scores['rouge1'].fmeasure
            r2 = rouge_scores['rouge2'].fmeasure
            rL = rouge_scores['rougeL'].fmeasure
            
        results.append({
            'file': fname,
            'BLEU': round(bleu, 4),
            'ROUGE-1 F1': round(r1, 4),
            'ROUGE-2 F1': round(r2, 4),
            'ROUGE-L F1': round(rL, 4)
        })
        
    df = pd.DataFrame(results)
    avg_metrics = df[['BLEU', 'ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1']].mean().to_dict()
    avg_metrics = {k: round(v, 4) for k, v in avg_metrics.items()}
    
    if print_results:
        print("\n Results:")
        display(df.style.format({'BLEU': '{:.4f}', 'ROUGE-1 F1': '{:.4f}', 
                                 'ROUGE-2 F1': '{:.4f}', 'ROUGE-L F1': '{:.4f}'}))
        print("\n Mean metrics:")
        for k, v in avg_metrics.items():
            print(f"  {k}: {v}")
            
    return df, avg_metrics

In [17]:
df, avg_metrics = calculate_md_metrics('MD', 'PereStruct')


 Results:


,file,BLEU,ROUGE-1 F1,ROUGE-2 F1,ROUGE-L F1
0,103.jpg.md,0.9308,0.9442,0.8759,0.9333
1,110.jpg.md,0.9830,0.9818,0.9693,0.9818
2,15.jpg.md,0.9679,0.8696,0.8060,0.6667
3,21.jpg.md,0.9465,0.9630,0.8944,0.9259
4,52.jpg.md,0.9568,0.9735,0.9554,0.9735
5,61.jpg.md,0.9866,1.0000,1.0000,1.0000
6,72.jpg.md,0.9495,0.9854,0.9412,0.9781
7,83.jpg.md,0.9397,0.9367,0.8846,0.9367
8,87.jpg.md,0.9524,0.9590,0.9333,0.8644
9,93.jpg.md,0.9741,1.0000,0.8621,0.9000



 Mean metrics:
  BLEU: 0.9587
  ROUGE-1 F1: 0.9613
  ROUGE-2 F1: 0.9122
  ROUGE-L F1: 0.916


# Qwen3.6-35B-A3B

In [18]:
df, avg_metrics = calculate_md_metrics('MD', 'Qwen3.6_35B_A3B')


 Results:


,file,BLEU,ROUGE-1 F1,ROUGE-2 F1,ROUGE-L F1
0,103.jpg.md,0.0075,0.0875,0.0081,0.0743
1,110.jpg.md,0.1842,0.6522,0.4118,0.5072
2,15.jpg.md,0.4718,0.7619,0.5246,0.6349
3,21.jpg.md,0.0048,0.0292,0.0073,0.0268
4,52.jpg.md,0.0808,0.4868,0.0397,0.2237
5,61.jpg.md,0.0142,0.0952,0.0302,0.0752
6,72.jpg.md,0.0799,0.3846,0.1466,0.2393
7,83.jpg.md,0.0114,0.2281,0.0357,0.1754
8,87.jpg.md,0.0927,0.4818,0.2353,0.3358
9,93.jpg.md,0.2419,0.7273,0.3019,0.5455



 Mean metrics:
  BLEU: 0.1189
  ROUGE-1 F1: 0.3935
  ROUGE-2 F1: 0.1741
  ROUGE-L F1: 0.2838


# Qwen3.6-Plus

In [26]:
df, avg_metrics = calculate_md_metrics('MD', 'Qwen3.6_Plus')


 Results:


,file,BLEU,ROUGE-1 F1,ROUGE-2 F1,ROUGE-L F1
0,103.jpg.md,0.1053,0.3019,0.0363,0.1981
1,110.jpg.md,0.0830,0.4084,0.1270,0.3141
2,15.jpg.md,0.8176,0.8750,0.7742,0.6875
3,21.jpg.md,0.2018,0.6607,0.3174,0.3810
4,52.jpg.md,0.2811,0.6233,0.2723,0.4093
5,61.jpg.md,0.3630,0.6761,0.4058,0.6761
6,72.jpg.md,0.4652,0.6087,0.4108,0.4147
7,83.jpg.md,0.1890,0.5968,0.2951,0.4677
8,87.jpg.md,0.3280,0.7405,0.5538,0.6718
9,93.jpg.md,0.5342,0.7931,0.4643,0.6552



 Mean metrics:
  BLEU: 0.3368
  ROUGE-1 F1: 0.6284
  ROUGE-2 F1: 0.3657
  ROUGE-L F1: 0.4875
